# Lab 04: Stripe Workbench

**Duration**: ~20 minutes  

## Learning Objectives

By the end of this notebook, you will:
- Know what Workbench is and how it fits into your debugging workflow
- Create a product, price, and Checkout Session using the Workbench Shell
- Simulate a spike in failed payments using test cards
- Investigate failures using the Events, Logs, and Health tabs
- Configure a webhook destination for proactive monitoring
- Explore Blueprints for ready-to-use integration examples

---

> **This lab is Dashboard-first.** Most steps are performed inside [Stripe Workbench](https://dashboard.stripe.com/workbench). This notebook provides the scenario context, the test data you need, and guided commentary to connect what you see in the UI to the underlying concepts.

## What is Workbench?

**Workbench** is a developer toolkit built directly into the Stripe Dashboard. It gives you a single place to:

| Tab | What you can do |
|-----|-----------------|
| **Shell** | Run Stripe CLI commands and API calls interactively |
| **API Explorer** | Build and tweak API requests without writing code |
| **Inspector** | Drill into any Stripe object as JSON |
| **Events** | Browse and filter all events in real time |
| **Logs** | See every API request your integration made |
| **Webhooks** | Monitor deliveries and add new event destinations |
| **Health** | Detect integration issues proactively |
| **Blueprints** | Pre-built, runnable integration examples |

### The Scenario

Your monitoring system has just fired an alert: **there's a spike in failed payments**. Customers haven't all complained yet — but backend errors are piling up. Your job is to investigate, identify the root cause, and set up proactive monitoring.

This notebook walks you through that investigation using Workbench.

---

## Setup: Connect to Stripe

In [ ]:
!pip install stripe --quiet

import stripe
import time
from datetime import datetime

try:
    from google.colab import userdata
    stripe.api_key = userdata.get('STRIPE_SECRET_KEY')
    print("Loaded API key from Colab Secrets.")
except Exception:
    import getpass
    stripe.api_key = getpass.getpass("Paste your Stripe test secret key (sk_test_...): ")

try:
    account = stripe.Account.retrieve()
    print(f"Connected: {account.id}")
except stripe.error.AuthenticationError:
    print("ERROR: Invalid API key.")

---

## Step 1: Open Workbench

1. Go to the [Stripe Dashboard](https://dashboard.stripe.com/test/dashboard)
2. Click **Developers** in the top navigation
3. Click **Workbench** — or press the **`` ` ``** key anywhere in the Dashboard to toggle it

You'll see the Shell panel on the left and the API Explorer on the right.

### Explore the Shell

The Shell runs Stripe CLI commands in your sandbox. Type `stripe` and press Enter to see available commands.

---

## Step 2: Create a Product with the Shell

In the Workbench Shell, run the following command to create a product with a default price:

```bash
stripe products create \
  --name "Workbench Demo Product" \
  --default-price-data.currency eur \
  --default-price-data.unit-amount 2000
```

**Observe:**
- The API Explorer panel on the right auto-populates with the parameters
- The response shows the full Product JSON object
- Click any object in the Inspector to drill into its relationships

**Note the `default_price` ID** from the response — you'll use it in Step 3.

> Alternatively, run the cell below to do the same thing via the Python SDK, then use the price ID in the Shell.

In [ ]:
product = stripe.Product.create(
    name="Workbench Demo Product",
    default_price_data={
        "currency": "eur",
        "unit_amount": 2000,
    }
)

PRICE_ID = product.default_price

print(f"Product ID: {product.id}")
print(f"Price ID:   {PRICE_ID}")
print()
print("Copy the Price ID above — you'll need it in Step 3.")

---

## Step 3: Create a Checkout Session

A Checkout Session generates a hosted payment page. In the Shell, replace `price_xxxxx` with your actual price ID:

```bash
stripe checkout sessions create \
  -d "line_items[0][price]"=price_xxxxx \
  -d "line_items[0][quantity]"=1 \
  --mode=payment \
  --success-url="https://dashboard.stripe.com" \
  --cancel-url="https://dashboard.stripe.com"
```

The response contains a `url` field — **open that URL** in a new tab to reach the hosted checkout page.

Or use the cell below:

In [ ]:
session = stripe.checkout.Session.create(
    line_items=[{"price": PRICE_ID, "quantity": 1}],
    mode="payment",
    success_url="https://dashboard.stripe.com",
    cancel_url="https://dashboard.stripe.com",
)

print(f"Checkout Session: {session.id}")
print(f"\nOpen this URL to reach the hosted checkout page:")
print(f"  {session.url}")
print()
print("Note: Behind the scenes this created a PaymentIntent, Customer, and Charge object.")

---

## Step 4: Simulate Payments — Including Failures

On the hosted checkout page, complete the form using each card below. Use any future expiry, any CVC, and any billing ZIP.

| Card number | What happens |
|-------------|-------------|
| `4000 0000 0000 9979` | Stolen card — declined |
| `4000 0000 0000 0119` | Processing error — declined |
| `4000 0000 0000 9987` | Lost card — declined |
| `4242 4242 4242 4242` | Visa — succeeds |

> You need to create a new Checkout Session for each payment attempt. Re-run the cell above to get a fresh session URL.

### Why This Matters

Some errors (e.g., lost card, stolen card) display a generic **"Your card was declined"** message to the customer. The specific decline code — `lost_card`, `stolen_card`, `processing_error` — is only visible to you in the backend. Without Workbench, you might only discover these failures from customer complaints.

Alternatively, simulate the failures in Python:

In [ ]:
# Simulate the same payment scenarios programmatically
test_cards = [
    ("Stolen card",       "pm_card_visa_chargeDeclinedStolenCard"),
    ("Processing error",  "pm_card_chargeDeclinedProcessingError"),
    ("Lost card",         "pm_card_visa_chargeDeclinedLostCard"),
    ("Success",           "pm_card_visa"),
]

payment_ids = []

for label, pm in test_cards:
    try:
        pi = stripe.PaymentIntent.create(
            amount=2000,
            currency="eur",
            payment_method=pm,
            confirm=True,
            automatic_payment_methods={"enabled": True, "allow_redirects": "never"}
        )
        payment_ids.append(pi.id)
        print(f"  {label:<20} → {pi.id} | status: {pi.status}")
    except stripe.error.CardError as e:
        print(f"  {label:<20} → DECLINED | decline_code: {e.error.decline_code}")

---

## Step 5: Investigate with Workbench — Events Tab

Now let's use Workbench to investigate what happened.

1. **Open Workbench** and click the **Events** tab
2. **Filter** by event type: select `payment_intent.payment_failed`
3. **Click any failed event** to expand its JSON payload
4. **Find `last_payment_error`** in the payload — this contains:
   - `decline_code`: the specific reason (`stolen_card`, `lost_card`, `processing_error`)
   - `message`: a description of the error
   - `code`: the high-level error code (`card_declined`)

This is the key insight: the customer only saw a generic decline message, but you can see exactly why it failed.

Retrieve the same information via the API:

In [ ]:
# Retrieve recent payment_intent.payment_failed events and inspect their errors
failed = stripe.Event.list(type="payment_intent.payment_failed", limit=5)

print(f"Recent payment failures:\n")
for e in failed.data:
    pi = e.data.object
    err = pi.last_payment_error
    created = datetime.fromtimestamp(e.created).strftime('%H:%M:%S')
    if err:
        print(f"  {created} | {pi.id}")
        print(f"    code:         {err.code}")
        print(f"    decline_code: {err.decline_code}")
        print(f"    message:      {err.message}")
        print()

---

## Step 6: Investigate with Workbench — Logs Tab

The **Logs** tab shows every API request your integration made, including:
- The full request body
- The full response (including errors)
- HTTP status codes
- Request latency

1. Open the **Logs** tab in Workbench
2. Filter by status: **Failed** (4xx / 5xx)
3. Click any failed request to see the full request and response
4. Compare: the request body shows `payment_method`, the response shows the `decline_code`

This is especially useful when you receive an error but can't reproduce it — the Logs show exactly what was sent and received.

---

## Step 7: Check the Health Tab

The **Health** tab proactively surfaces integration issues — before your customers notice.

1. Click the **Health** tab in Workbench
2. Review any alerts — you may see a spike in payment failures from this exercise
3. Click an alert to see the affected events and suggested remediation

Health alerts can flag:
- Unusual decline rate increases
- Webhook delivery failures
- API error rate spikes
- Deprecated API usage

---

## Step 8: Add a Webhook for Proactive Monitoring

Instead of checking the Dashboard manually, configure a webhook so your system is notified the moment a payment fails.

### Via the Dashboard

1. Open the **Webhooks** tab in Workbench (or go to [Developers → Webhooks](https://dashboard.stripe.com/test/webhooks))
2. Click **Add destination**
3. Choose the event: `payment_intent.payment_failed`
4. Enter your endpoint URL and save

### Via the API

The cell below registers a webhook programmatically — the same approach covered in Lab 02.

In [ ]:
# Create a webhook endpoint for payment failure monitoring
# Replace the URL with your actual endpoint in production
MONITORING_URL = "https://your-server.example.com/stripe/webhooks"

print("To register a real monitoring endpoint, replace MONITORING_URL above and uncomment:")
print()
print("""
endpoint = stripe.WebhookEndpoint.create(
    url=MONITORING_URL,
    enabled_events=[
        "payment_intent.payment_failed",
        "charge.dispute.created",
        "invoice.payment_failed",
    ],
    description="Payment failure monitoring"
)
print(f"Webhook registered: {endpoint.id}")
print(f"Signing secret:     {endpoint.secret}")
""")

---

## Step 9: Explore Blueprints (Optional)

**Blueprints** are pre-built, runnable integration examples for common workflows.

1. Open the **Blueprints** section in Workbench
2. Browse available workflows (one-time payment, subscription, invoicing, etc.)
3. Select one and run it step by step — each step shows the API call and its result
4. Blueprints are a fast way to understand how multiple API objects relate to each other

Useful for:
- Learning a new integration pattern
- Generating sample data for testing
- Sharing reproducible examples with your team or Stripe support

---

## Putting It All Together

Let's query the full picture of what happened in this exercise.

In [ ]:
# Summary of payment attempts in this session
events = stripe.Event.list(limit=20)

succeeded = []
failed    = []

for e in events.data:
    if e.type == 'payment_intent.succeeded':
        succeeded.append(e)
    elif e.type == 'payment_intent.payment_failed':
        failed.append(e)

print(f"Payment attempts (last 20 events):")
print(f"  Succeeded: {len(succeeded)}")
print(f"  Failed:    {len(failed)}")
if len(succeeded) + len(failed) > 0:
    failure_rate = len(failed) / (len(succeeded) + len(failed)) * 100
    print(f"  Failure rate: {failure_rate:.0f}%")

if failed:
    print(f"\nFailure breakdown:")
    from collections import Counter
    decline_codes = Counter()
    for e in failed:
        err = e.data.object.last_payment_error
        if err and err.decline_code:
            decline_codes[err.decline_code] += 1
    for code, count in decline_codes.most_common():
        print(f"  {code}: {count}")

---

## Summary

| Workbench Tool | Used for |
|----------------|----------|
| **Shell** | Interactive API calls and quick experiments without writing code |
| **API Explorer** | Build requests with auto-complete — no JSON editing |
| **Inspector** | Drill into objects and navigate their relationships |
| **Events** | Filter, inspect, and replay events — including `last_payment_error` |
| **Logs** | Full request/response history for every API call |
| **Health** | Proactive alerts for integration issues |
| **Webhooks** | Monitor delivery status and add event destinations |
| **Blueprints** | Pre-built, step-by-step integration examples |

### Key Takeaway

**Decline codes are only visible in the backend.** Customers see a generic message, but Workbench's Events tab shows you the specific `decline_code` (`stolen_card`, `processing_error`, etc.) so you can investigate, understand patterns, and take action — before customers complain.

## Workshop Complete!

You've completed all four labs:

| Lab | Topic |
|-----|-------|
| 01 | API Testing: Sandboxes, Test Cards, Test Clocks |
| 02 | Webhook Integration: Build, Test, and Troubleshoot |
| 03 | Infrastructure as Code: Terraform + Stripe Provider |
| 04 | Workbench: Debug, Inspect, and Monitor |

### Resources

- [Workbench docs](https://docs.stripe.com/workbench)
- [Stripe Testing reference](https://docs.stripe.com/testing)
- [Stripe Terraform Provider](https://registry.terraform.io/providers/stripe/stripe/latest/docs)
- [Webhook best practices](https://docs.stripe.com/webhooks/best-practices)
- [Stripe Samples on GitHub](https://github.com/stripe-samples)